In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline,AutoTokenizer,AutoModelForSeq2SeqLM


In [11]:


class OfflineSentenceFusion:

    def __init__(self):

        # semantic model
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

        # T5 model (ONLY ONCE)
        self.tokenizer = AutoTokenizer.from_pretrained("t5-small")
        self.model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

    # embeddings
    def embed(self, sentences):
        return self.embedder.encode(sentences)

    # similarity
    def similarity_matrix(self, embeddings):
        return cosine_similarity(embeddings)

    # select important sentences
    def select_core_sentences(self, sentences, sim_matrix, top_k=3):
        scores = sim_matrix.sum(axis=1)
        idx = np.argsort(scores)[::-1]
        return [sentences[i] for i in idx[:top_k]]

    # fuse sentences
    def fuse_text(self, sentences):
        return ". ".join(sentences)

    # SINGLE rewrite (IMPORTANT FIXED)
    def rewrite(self, text):

        input_text = "summarize: " + text.strip()

        inputs = self.tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )

        outputs = self.model.generate(
            **inputs,
            max_length=80,
            min_length=20,
            num_beams=5,
            repetition_penalty=2.0,
            no_repeat_ngram_size=3
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    # FINAL PIPELINE
    def run(self, sentences):

        embeddings = self.embed(sentences)

        sim = self.similarity_matrix(embeddings)

        core = self.select_core_sentences(sentences, sim)

        fused = self.fuse_text(core)

        final = self.rewrite(fused)

        return final

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
import torch
sentences =[
    "I am Feroz Ahmmed",
    "I am 29 years old",
    "I am 5 feet 11 inches",
    "I am 110KG",
    "I am  a software developer"
]
text = "Combine into a single natural sentence: ".join(sentences)

inputs = tokenizer(
    text,
    max_length=512,
    truncation=True,
    return_tensors="pt"
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    min_length=10,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

output = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(output)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

I want to place an order. Order details - Chhiken pizaa - Hot tea. Bottle of mineral water - Takeway request.
